# 库存路径问题 (IRP)

**类别：** 路径规划

来源：[https://www.hexaly.com/templates/inventory-routing-problem-irp](https://www.hexaly.com/templates/inventory-routing-problem-irp)


## 问题

**库存路径问题 (IRP)** 是一个配送问题，其中需要在给定的时间范围内将产品从一个供应商运送到多个客户。每个客户都有一个最大库存水平。供应商监控每个客户的库存并确定其补货策略，保证客户不会出现缺货（供应商管理库存策略）。从供应商到客户的运输由具有给定容量的车辆完成。

	

### 学到的建模原则

- 添加 [list decision variables](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 来建模卡车的客户序列
- 定义 [lambda functions](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来计算每个时间槽的行驶距离
- 递归地定义表达式来计算库存水平


## 数据

我们提供的库存路径问题 (IRP) 实例来自 [Archetti et al. instances](https://www.leandro-coelho.com/instances/inventory-routing/)。数据文件的格式如下：

- 第一行：客户数、规划时间范围内的离散时间槽数以及车辆的运输容量。
- 第二行：对于供应商：

- 供应商的索引
- x 坐标
- y 坐标
- 供应商库存的起始水平
- 每个时间槽可获得的产品数量
- 单位库存成本
- 接下来的每一行：对每个客户

- 客户的索引
- x 坐标
- y 坐标
- 库存的起始水平
- 最大库存水平
- 最小库存水平
- 每个时间槽消耗的产品数量
- 单位库存成本


## 模型

库存路径问题 (IRP) 的Hexaly模型使用 [list decision variables](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html)。对每个离散时间槽，我们定义一个 list variable，表示卡车在该时间槽访问的客户。每条路线的成本取决于卡车行驶的距离。使用 [lambda function](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html)，我们将沿途从一个客户到下一个客户的距离累加起来。

我们递归地计算供应商的库存水平。事实上，时间 t 的库存水平等于时间 t-1 的库存水平，加上时间 t-1 可获得的产品数量，再减去时间 t-1 运送给客户的产品总量。供应商处的缺货约束确保供应商的库存水平始终足以运送给定时间槽内交付给客户的总数量。

类似地，我们随后计算客户的库存水平。对于每个客户，时间 t 的库存水平等于时间 t-1 的库存水平，加上时间 t-1 从供应商运送给该客户的产品数量，再减去时间 t-1 消耗的产品数量。客户处的缺货约束保证每个客户的库存水平始终为正。

容量约束确保车辆装载的产品总量不超过其容量。最大水平约束保证从供应商运送给每个客户的产品数量不超过该客户的最大库存水平。

目标是最小化所有成本之和：供应商的总库存成本、客户的总库存成本以及总运输成本。


## Results

**Hexaly Optimizer 在 36% 的 IRP 实例上改进了文献中的最优已知解**，这些实例来自 2016 ROADEF/EURO Challenge。我们的 [Inventory Routing Problem (IRP) benchmark page](https://www.hexaly.com/benchmark/hexaly-establishes-new-records-for-the-inventory-routing-problem-irp) 展示了 Hexaly Optimizer 在这一富有挑战性问题上的性能。

[Explore this benchmark](https://www.hexaly.com/benchmark/hexaly-establishes-new-records-for-the-inventory-routing-problem-irp)


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys
import math


def read_elem(filename):
    with open(filename) as f:
        return [str(elem) for elem in f.read().split()]


def main(instance_file, str_time_limit, sol_file):
    #
    # Read instance data
    #
    nb_customers, horizon_length, capacity, start_level_supplier, production_rate_supplier, \
        holding_cost_supplier, start_level, max_level, demand_rate, holding_cost, \
        dist_matrix_data, dist_supplier_data = read_input_irp(instance_file)

    with hexaly.optimizer.HexalyOptimizer() as optimizer:
        #
        # Declare the optimization model
        #
        model = optimizer.model

        # Quantity of product delivered at each discrete time instant of
        # the planning time horizon to each customer
        delivery = [[model.float(0, capacity) for i in range(nb_customers)]
                    for _ in range(horizon_length)]

        # Sequence of customers visited at each discrete time instant of
        # the planning time horizon
        route = [model.list(nb_customers) for t in range(horizon_length)]

        # Customers receive products only if they are visited
        is_delivered = [[model.contains(route[t], i) for i in range(nb_customers)]
                        for t in range(horizon_length)]

        # Create Hexaly arrays to be able to access them with an "at" operator
        dist_matrix = model.array(dist_matrix_data)
        dist_supplier = model.array(dist_supplier_data)

        dist_routes = [None for _ in range(horizon_length)]

        for t in range(horizon_length):
            sequence = route[t]
            c = model.count(sequence)

            # Distance traveled at instant t
            dist_lambda = model.lambda_function(
                lambda i:
                    model.at(
                        dist_matrix,
                        sequence[i - 1],
                        sequence[i]))
            dist_routes[t] = model.iif(
                c > 0,
                dist_supplier[sequence[0]]
                + model.sum(model.range(1, c), dist_lambda)
                + dist_supplier[sequence[c - 1]],
                0)

        # Stockout constraints at the supplier
        inventory_supplier = [None for _ in range(horizon_length + 1)]
        inventory_supplier[0] = start_level_supplier
        for t in range(horizon_length):
            inventory_supplier[t + 1] = inventory_supplier[t] - model.sum(
                delivery[t][i] for i in range(nb_customers)) + production_rate_supplier
            model.constraint(inventory_supplier[t] >= model.sum(delivery[t][i] for i in range(nb_customers)))

        # Stockout constraints at the customers
        inventory = [[None for _ in range(horizon_length + 1)] for _ in range(nb_customers)]
        for i in range(nb_customers):
            inventory[i][0] = start_level[i]
            for t in range(horizon_length):
                inventory[i][t + 1] = inventory[i][t] + delivery[t][i] - demand_rate[i]
                model.constraint(inventory[i][t + 1] >= 0)

        for t in range(horizon_length):
            # Capacity constraints
            model.constraint(
                model.sum((delivery[t][i]) for i in range(nb_customers)) <= capacity)

            # Maximum level constraints
            for i in range(nb_customers):
                model.constraint(delivery[t][i] <= max_level[i] - inventory[i][t])
                model.constraint(delivery[t][i] <= max_level[i] * is_delivered[t][i])

        # Total inventory cost at the supplier
        total_cost_inventory_supplier = holding_cost_supplier * model.sum(
            inventory_supplier[t] for t in range(horizon_length + 1))

        # Total inventory cost at customers
        total_cost_inventory = model.sum(model.sum(
            holding_cost[i] * inventory[i][t] for t in range(horizon_length + 1))
            for i in range(nb_customers))

        # Total transportation cost
        total_cost_route = model.sum(dist_routes[t] for t in range(horizon_length))

        # Objective: minimize the sum of all costs
        objective = total_cost_inventory_supplier + total_cost_inventory + total_cost_route
        model.minimize(objective)

        model.close()

        # Parameterize the optimizer
        optimizer.param.time_limit = int(str_time_limit)

        optimizer.solve()

        #
        # Write the solution in a file with the following format :
        # - total distance run by the vehicle
        # - the nodes visited at each time step (omitting the start/end at the supplier)
        #
        if len(sys.argv) >= 3:
            with open(sol_file, 'w') as f:
                f.write("%d\n" % (total_cost_route.value))
                for t in range(horizon_length):
                    for customer in route[t].value:
                        f.write("%d " % (customer + 1))
                    f.write("\n")


# The input files follow the "Archetti" format
def read_input_irp(filename):
    file_it = iter(read_elem(filename))

    nb_customers = int(next(file_it)) - 1
    horizon_length = int(next(file_it))
    capacity = int(next(file_it))

    x_coord = [None] * nb_customers
    y_coord = [None] * nb_customers
    start_level = [None] * nb_customers
    max_level = [None] * nb_customers
    min_level = [None] * nb_customers
    demand_rate = [None] * nb_customers
    holding_cost = [None] * nb_customers

    next(file_it)
    x_coord_supplier = float(next(file_it))
    y_coord_supplier = float(next(file_it))
    start_level_supplier = int(next(file_it))
    production_rate_supplier = int(next(file_it))
    holding_cost_supplier = float(next(file_it))
    for i in range(nb_customers):
        next(file_it)
        x_coord[i] = float(next(file_it))
        y_coord[i] = float(next(file_it))
        start_level[i] = int(next(file_it))
        max_level[i] = int(next(file_it))
        min_level[i] = int(next(file_it))
        demand_rate[i] = int(next(file_it))
        holding_cost[i] = float(next(file_it))

    distance_matrix = compute_distance_matrix(x_coord, y_coord)
    distance_supplier = compute_distance_supplier(x_coord_supplier, y_coord_supplier, x_coord, y_coord)

    return nb_customers, horizon_length, capacity, start_level_supplier, \
        production_rate_supplier, holding_cost_supplier, start_level, max_level, \
        demand_rate, holding_cost, distance_matrix, distance_supplier


# Compute the distance matrix
def compute_distance_matrix(x_coord, y_coord):
    nb_customers = len(x_coord)
    distance_matrix = [[None for i in range(nb_customers)] for j in range(nb_customers)]
    for i in range(nb_customers):
        distance_matrix[i][i] = 0
        for j in range(nb_customers):
            dist = compute_dist(x_coord[i], x_coord[j], y_coord[i], y_coord[j])
            distance_matrix[i][j] = dist
            distance_matrix[j][i] = dist
    return distance_matrix


# Compute the distances to the supplier
def compute_distance_supplier(x_coord_supplier, y_coord_supplier, x_coord, y_coord):
    nb_customers = len(x_coord)
    distance_supplier = [None] * nb_customers
    for i in range(nb_customers):
        dist = compute_dist(x_coord_supplier, x_coord[i], y_coord_supplier, y_coord[i])
        distance_supplier[i] = dist
    return distance_supplier


def compute_dist(xi, xj, yi, yj):
    return round(math.sqrt(math.pow(xi - xj, 2) + math.pow(yi - yj, 2)))


if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python irp.py input_file [output_file] [time_limit]")
        sys.exit(1)

    instance_file = sys.argv[1]
    sol_file = sys.argv[2] if len(sys.argv) > 2 else None
    str_time_limit = sys.argv[3] if len(sys.argv) > 3 else "20"

    main(instance_file, str_time_limit, sol_file)
